# Scopus Author IDs of Nobel laureates (free API tier)

This version works **without institutional Scopus access**. It only needs a free Elsevier API key, which anyone can create at https://dev.elsevier.com/.

The free tier does not include the Scopus Author Search API, and the free Scopus Search view does not return author IDs. It does include the **Abstract Retrieval API** (`META` view), which lists the Scopus Author IDs of a paper's authors. So instead of searching for the person, we look at their papers:

1. **Wikidata** – many laureates already have a curated Scopus Author ID ([P1153](https://www.wikidata.org/wiki/Property:P1153)), plus ORCID ([P496](https://www.wikidata.org/wiki/Property:P496)) and OpenAlex ID ([P10283](https://www.wikidata.org/wiki/Property:P10283)). No key needed.
2. **OpenAlex** ([openalex.org](https://openalex.org), free and open) – find the laureate's author profile (via the OpenAlex ID or ORCID from Wikidata, or by name) and pick a few of their papers with a DOI, preferring papers where they are **first author**.
3. **Scopus Abstract Retrieval** – for each DOI, read the Scopus Author IDs of the authors and keep the one whose name matches the laureate. The ID that recurs across papers is the laureate's Scopus ID (a simple vote).

## Prerequisites

- A free Elsevier API key in the `ELSEVIER_API_KEY` environment variable (or in a `.env` file).
- Optional: `OPENALEX_API_KEY` (free, from https://openalex.org) and `OPENALEX_MAILTO` (your e-mail, for OpenAlex's polite pool).
- `pip install -r requirements.txt`

> **Notes.** The free Elsevier tier is limited and its terms restrict usage to non-commercial purposes; check https://dev.elsevier.com/policy.html. If the Abstract Retrieval API returns 401/403 for your key, the free tier is not enough and you need the institutional version (`nobel_scopus_ids.ipynb`). Peace and Literature laureates, some Economics laureates, and scientists whose output predates the 1970s often have no Scopus profile; an empty result for them is expected.

In [1]:
import json
import os
import re
import time
import hashlib
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

from collections import Counter

import pandas as pd
import requests
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Set the input file path and the column names. `.csv`, `.xlsx` and `.txt` (one name per line) are supported.

If the list also has the prize **category** and **award year**, matching against Wikidata is restricted to the laureates of that category and year, which makes it robust to spelling variants, transliterations and missing middle names. Set `CATEGORY_COLUMN` / `YEAR_COLUMN` to `None` if your list does not have them.

In [2]:
INPUT_FILE = Path("data/nobel_laureates.csv")   # your list of names
NAME_COLUMN = "name"                            # column with the names (ignored for .txt)
CATEGORY_COLUMN = "category"                    # e.g. "Physics", "Chemistry", "Physiology or Medicine", "Economics"
YEAR_COLUMN = "award_year"
OUTPUT_FILE = Path("output/nobel_scopus_ids_wikidata.csv")

ELSEVIER_API_KEY = os.getenv("ELSEVIER_API_KEY")
OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY")    # optional
OPENALEX_MAILTO = os.getenv("OPENALEX_MAILTO")      # optional, your e-mail

CACHE_DIR = Path("cache")          # API responses are cached to save quota
SCOPUS_SLEEP = 0.25                # pause between Scopus calls (limit ~2-3 req/s)
FUZZY_THRESHOLD = 0.92             # minimum similarity for matching names to Wikidata
N_PAPERS = 5                       # papers per laureate checked in Scopus

USER_AGENT = "nobel_oa/0.1 (https://github.com/gavieira/nobel_oa)"

CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
print("Elsevier API key set:", bool(ELSEVIER_API_KEY))

Elsevier API key set: False


## Load the list of names

In [3]:
def load_names(path, column=NAME_COLUMN, category_column=CATEGORY_COLUMN, year_column=YEAR_COLUMN):
    path = Path(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".txt":
        df = pd.DataFrame({column: path.read_text(encoding="utf-8").splitlines()})
    else:
        df = pd.read_csv(path)
    df = df.rename(columns={column: "name", category_column: "category", year_column: "award_year"})
    df = df.dropna(subset=["name"])
    df["name"] = df["name"].astype(str).str.split().str.join(" ")
    df = df[df["name"].ne("")]
    if "award_year" in df:
        df["award_year"] = pd.to_numeric(df["award_year"], errors="coerce").astype("Int64")
    # the same person may appear twice (e.g. Sharpless, 2001 and 2022): keep one row per award
    return df.drop_duplicates().reset_index(drop=True)

names = load_names(INPUT_FILE)
print(len(names), "names loaded")
names.head()

257 names loaded


,category,award_year,name
0,Physiology or Medicine,2000,Arvid Carlsson
1,Physiology or Medicine,2000,Paul Greengard
2,Physiology or Medicine,2000,Eric Richard Kandel
3,Physiology or Medicine,2001,Leland Harrison Lee Hartwell
4,Physiology or Medicine,2001,Paul Maxime Nurse


## Name normalization

To match names coming from different sources: strip accents, punctuation, dotted initials and suffixes (`Jr.`, `III`…), and convert `"Last, First"` to `"First Last"`.

In [4]:
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "sir", "dr", "prof", "lord", "baron"}

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def reorder(name):
    # "Curie, Marie" -> "Marie Curie" (only when there is exactly one comma)
    if name.count(",") == 1:
        last, first = [p.strip() for p in name.split(",")]
        if first.lower().strip(".") not in SUFFIXES:
            return f"{first} {last}"
    return name

def normalize_name(name):
    s = strip_accents(reorder(str(name))).lower()
    s = re.sub(r"[^a-z\s-]", " ", s).replace("-", " ")
    tokens = [t for t in s.split() if t not in SUFFIXES]
    return " ".join(tokens)

def name_key(name):
    # lenient key: first token + last token (ignores middle names)
    tokens = normalize_name(name).split()
    return f"{tokens[0]} {tokens[-1]}" if len(tokens) > 1 else " ".join(tokens)

def split_name(name):
    # returns (given name, surname) for the Scopus search
    tokens = strip_accents(reorder(str(name))).replace(".", " ").split()
    tokens = [t for t in tokens if t.lower().strip(",") not in SUFFIXES]
    if len(tokens) == 1:
        return "", tokens[0]
    # surname particles: "van der Waals", "de Broglie", "von Klitzing"
    particles = {"van", "von", "der", "de", "da", "di", "du", "la", "le", "del", "dos", "das", "ten", "ter"}
    i = len(tokens) - 1
    while i > 1 and tokens[i - 1].lower() in particles:
        i -= 1
    return " ".join(tokens[:i]), " ".join(tokens[i:])

for n in ["Curie, Marie", "Louis de Broglie", "Johannes Diderik van der Waals", "Martin Luther King Jr.", "Gabriel García Márquez"]:
    print(f"{n!r:35} -> {normalize_name(n)!r:30} key={name_key(n)!r:22} split={split_name(n)}")

'Curie, Marie'                      -> 'marie curie'                  key='marie curie'          split=('Marie', 'Curie')
'Louis de Broglie'                  -> 'louis de broglie'             key='louis broglie'        split=('Louis', 'de Broglie')
'Johannes Diderik van der Waals'    -> 'johannes diderik van der waals' key='johannes waals'       split=('Johannes Diderik', 'van der Waals')
'Martin Luther King Jr.'            -> 'martin luther king'           key='martin king'          split=('Martin Luther', 'King')
'Gabriel García Márquez'            -> 'gabriel garcia marquez'       key='gabriel marquez'      split=('Gabriel Garcia', 'Marquez')


## Cache helper

Every HTTP response is stored in `cache/`, so re-running the notebook does not consume API quota.

In [5]:
def cached_get(url, params=None, headers=None, namespace="http", timeout=60, proxies=None):
    key = hashlib.sha1(json.dumps([url, params], sort_keys=True).encode()).hexdigest()
    path = CACHE_DIR / namespace / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8")), True
    for attempt in range(5):
        r = requests.get(url, params=params, headers=headers, timeout=timeout, proxies=proxies)
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        data = r.json()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(data), encoding="utf-8")
        return data, False
    r.raise_for_status()

## Step 1 – Wikidata

A single query returns every person who received a Nobel Prize (including the Economics prize, which is a separate item in Wikidata), with the award year, labels, alternative names, Scopus ID and ORCID.

In [6]:
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

QUERY = '''
SELECT ?person ?label ?alias ?scopus ?orcid ?openalex ?awardLabel ?date WHERE {
  # Physics, Chemistry, Physiology or Medicine, Economics, Literature, Peace
  VALUES ?award { wd:Q38104 wd:Q44585 wd:Q80061 wd:Q47170 wd:Q37922 wd:Q35637 }
  ?person p:P166 ?st .
  ?st ps:P166 ?award .
  OPTIONAL { ?st pq:P585 ?date }                # award year
  ?person wdt:P31 wd:Q5 .                       # people only (excludes organizations)
  ?person rdfs:label ?label .
  FILTER(LANG(?label) IN ("en", "mul"))
  OPTIONAL { ?person skos:altLabel ?alias . FILTER(LANG(?alias) = "en") }
  OPTIONAL { ?person wdt:P1153 ?scopus }
  OPTIONAL { ?person wdt:P496 ?orcid }
  OPTIONAL { ?person wdt:P10283 ?openalex }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". ?award rdfs:label ?awardLabel . }
}
'''

data, from_cache = cached_get(
    WIKIDATA_SPARQL,
    params={"query": QUERY, "format": "json"},
    headers={"User-Agent": USER_AGENT, "Accept": "application/sparql-results+json"},
    namespace="wikidata",
)

def binding_value(row, key):
    return row.get(key, {}).get("value")

wd_raw = pd.DataFrame([
    {k: binding_value(row, k) for k in ["person", "label", "alias", "scopus", "orcid", "openalex", "awardLabel", "date"]}
    for row in data["results"]["bindings"]
])
wd_raw["qid"] = wd_raw["person"].str.rsplit("/", n=1).str[-1]

CATEGORY_KEYWORDS = {"physics": "Physics", "chemistry": "Chemistry", "medicine": "Physiology or Medicine",
                     "econom": "Economics", "literature": "Literature", "peace": "Peace"}

def canonical_category(text):
    text = str(text).lower()
    return next((v for k, v in CATEGORY_KEYWORDS.items() if k in text), None)

# one row per (person, category, year)
awards = (
    wd_raw.assign(category=wd_raw["awardLabel"].map(canonical_category),
                  award_year=pd.to_numeric(wd_raw["date"].str[:4], errors="coerce").astype("Int64"))
    [["qid", "category", "award_year"]].drop_duplicates()
)
print(awards.groupby("category")["qid"].nunique().to_string())

def join_unique(values):
    vals = sorted({v for v in values if isinstance(v, str) and v})
    return ";".join(vals) if vals else None

wikidata = (
    wd_raw.groupby("qid")
    .agg(wd_label=("label", "first"),
         wd_scopus_ids=("scopus", join_unique),
         wd_orcid=("orcid", join_unique),
         wd_openalex=("openalex", join_unique),
         wd_awards=("awardLabel", join_unique))
    .reset_index()
)
print(f"{len(wikidata)} laureates in Wikidata ({'cached' if from_cache else 'fresh query'});",
      f"{wikidata['wd_scopus_ids'].notna().sum()} with Scopus ID,",
      f"{wikidata['wd_orcid'].notna().sum()} with ORCID")
wikidata.head()

category
Chemistry                 198
Economics                  99
Literature                122
Peace                     112
Physics                   230
Physiology or Medicine    235
994 laureates in Wikidata (fresh query); 445 with Scopus ID, 197 with ORCID


,qid,wd_label,wd_scopus_ids,wd_orcid,wd_openalex,wd_awards
0,Q1002250,Duncan Haldane,6701402909,0000-0001-6420-9549,NaN,Nobel Prize in Physics
1,Q102034,Ronald Ross,NaN,NaN,NaN,Nobel Prize in Physiology or Medicine
2,Q102250,Peter Agre,7102695643,0000-0002-9581-890X,NaN,Nobel Prize in Chemistry
3,Q102266,Sidney Altman,NaN,0000-0001-7287-0337,NaN,Nobel Prize in Chemistry
4,Q102278,Christian B. Anfinsen,NaN,NaN,NaN,Nobel Prize in Chemistry


### Matching your names to Wikidata

Tried in order:

1. **Award** (when category and year are available): among the laureates of that category and year (at most three people), pick the one whose name tokens best match the input name. Tokens are compared approximately, so transliterations such as *Alfiorov*/*Alferov* or *Novosselov*/*Novoselov* still match.
2. **Exact**: normalized name identical to the Wikidata label or an alias.
3. **First+last**: same *first + last name* key, provided it points to a single person.
4. **Fuzzy**: whole-name similarity ≥ `FUZZY_THRESHOLD`.

In [7]:
# index: name form -> set of QIDs
name_forms = pd.concat([
    wd_raw[["qid", "label"]].rename(columns={"label": "form"}),
    wd_raw[["qid", "alias"]].rename(columns={"alias": "form"}),
]).dropna().drop_duplicates()
name_forms["norm"] = name_forms["form"].map(normalize_name)
name_forms["key"] = name_forms["form"].map(name_key)

by_norm = name_forms.groupby("norm")["qid"].agg(set).to_dict()
by_key = name_forms.groupby("key")["qid"].agg(set).to_dict()
norm_list = list(by_norm)

def match_wikidata(name):
    norm = normalize_name(name)
    if norm in by_norm and len(by_norm[norm]) == 1:
        return next(iter(by_norm[norm])), "exact", 1.0
    key = name_key(name)
    if key in by_key and len(by_key[key]) == 1:
        return next(iter(by_key[key])), "first+last", 1.0
    best, score = None, 0.0
    for cand in norm_list:
        s = SequenceMatcher(None, norm, cand).ratio()
        if s > score:
            best, score = cand, s
    if best and score >= FUZZY_THRESHOLD and len(by_norm[best]) == 1:
        return next(iter(by_norm[best])), "fuzzy", round(score, 3)
    ambiguous = by_norm.get(norm) or by_key.get(key)
    return None, ("ambiguous" if ambiguous else "no match"), round(score, 3)

TOKEN_THRESHOLD = 0.75   # minimum similarity for two name tokens to count as the same

forms_by_qid = name_forms.groupby("qid")["norm"].agg(list).to_dict()

def token_score(name_tokens, form_tokens):
    score = 0.0
    for t in name_tokens:
        if len(t) < 2:   # skip initials
            continue
        best = max((SequenceMatcher(None, t, f).ratio() for f in form_tokens), default=0.0)
        if best >= TOKEN_THRESHOLD:
            score += best
    return score

def match_by_award(name, category, year):
    group = awards.loc[(awards["category"] == canonical_category(category))
                       & (awards["award_year"] == year), "qid"].unique()
    if len(group) == 0:
        return None
    name_tokens = normalize_name(name).split()
    ranked = sorted(((max(token_score(name_tokens, f.split()) for f in forms_by_qid.get(q, [""])), q)
                     for q in group), reverse=True)
    best_score, best_qid = ranked[0]
    if best_score > 0 and (len(ranked) == 1 or best_score > ranked[1][0]):
        return best_qid, "award", round(best_score, 3)
    return None

def match_row(row):
    if pd.notna(row.get("category")) and pd.notna(row.get("award_year")):
        m = match_by_award(row["name"], row["category"], row["award_year"])
        if m:
            return m
    return match_wikidata(row["name"])

matches = names.apply(match_row, axis=1)
names["qid"] = matches.str[0]
names["wd_match"] = matches.str[1]
names["wd_match_score"] = matches.str[2]
result = names.merge(wikidata, on="qid", how="left")
result["wd_match"].value_counts()

wd_match
award    257
Name: count, dtype: int64

In [8]:
# check the fuzzy matches and the names that did not match
result.loc[result["wd_match"].isin(["fuzzy", "ambiguous", "no match"]),
           ["name", "wd_match", "wd_match_score", "wd_label", "qid"]]

,name,wd_match,wd_match_score,wd_label,qid


## Step 2 – OpenAlex: find the laureate and a few of their papers

API documentation: https://docs.openalex.org. The author profile is taken, in order of reliability, from the OpenAlex ID in Wikidata, the ORCID in Wikidata, or a name search (the most-cited profile whose name matches).

Papers are chosen so that the laureate is easy to spot in the Scopus author list: first-authored papers first, then papers with the fewest authors.

In [9]:
OPENALEX = "https://api.openalex.org"

def openalex_get(path, params=None):
    params = dict(params or {})
    if OPENALEX_API_KEY:
        params["api_key"] = OPENALEX_API_KEY
    if OPENALEX_MAILTO:
        params["mailto"] = OPENALEX_MAILTO
    try:
        data, _ = cached_get(f"{OPENALEX}/{path}", params=params,
                             headers={"User-Agent": USER_AGENT}, namespace="openalex")
        return data
    except requests.HTTPError as e:
        if e.response is not None and e.response.status_code == 404:
            return None
        raise

def short_id(openalex_id):
    return openalex_id.rsplit("/", 1)[-1] if openalex_id else None

def surname_matches(laureate_name, other_name, threshold=0.85):
    # the laureate's surname must match one of the tokens of the other name
    last = normalize_name(split_name(laureate_name)[1]).split()
    other = normalize_name(other_name).split()
    return bool(last) and all(
        max((SequenceMatcher(None, t, o).ratio() for o in other), default=0) >= threshold for t in last)

def given_initials(name):
    # first letters of the given names, e.g. "Thomas A. Steitz" -> {"t", "a"}
    return {t[0] for t in normalize_name(split_name(name)[0]).split() if t}

def matching_authors(laureate_names, authors):
    # authors whose surname matches; if several (e.g. Thomas and Joan Steitz on the same paper),
    # keep those whose first initial is one of the laureate's given-name initials
    hits = [a for a in authors if any(surname_matches(n, a["name"]) for n in laureate_names)]
    if len({a["auid"] for a in hits}) > 1:
        initials = set().union(*(given_initials(n) for n in laureate_names))
        narrowed = [a for a in hits if normalize_name(a["name"])[:1] in initials]
        hits = narrowed or hits
    return {a["auid"] for a in hits}

def find_openalex_author(row):
    if isinstance(row.get("wd_openalex"), str):
        a = openalex_get(f"authors/{row['wd_openalex'].split(';')[0]}")
        if a:
            return a, "wikidata"
    if isinstance(row.get("wd_orcid"), str):
        for orcid in row["wd_orcid"].split(";"):
            a = openalex_get(f"authors/orcid:{orcid}")
            if a:
                return a, "orcid"
    name = row["wd_label"] if isinstance(row.get("wd_label"), str) else row["name"]
    data = openalex_get("authors", {"search": name, "per_page": 25})
    cands = [a for a in (data or {}).get("results", []) if surname_matches(name, a.get("display_name", ""))]
    if cands:
        return max(cands, key=lambda a: a.get("cited_by_count", 0)), "name"
    return None, None

def pick_papers(author_id, n=N_PAPERS):
    data = openalex_get("works", {
        "filter": f"author.id:{short_id(author_id)},has_doi:true",
        "sort": "cited_by_count:desc",
        "per_page": 50,
        "select": "id,doi,publication_year,authorships",
    })
    papers = []
    for w in (data or {}).get("results", []):
        mine = [a for a in w.get("authorships", []) if (a.get("author") or {}).get("id") == author_id]
        if not mine or not w.get("doi"):
            continue
        papers.append({"doi": w["doi"].replace("https://doi.org/", ""),
                       "first_author": mine[0].get("author_position") == "first",
                       "n_authors": len(w.get("authorships", []))})
    papers.sort(key=lambda p: (not p["first_author"], p["n_authors"]))
    return papers[:n]

## Step 3 – Scopus Abstract Retrieval (free tier)

Documentation: https://dev.elsevier.com/documentation/AbstractRetrievalAPI.wadl. We request the `META` view by DOI and read the `authors` list; if a response only carries the first author (`dc:creator`), first-authored papers still give us the ID.

In [10]:
SCOPUS_ABSTRACT = "https://api.elsevier.com/content/abstract/doi/"

def as_list(x):
    return x if isinstance(x, list) else ([x] if x else [])

def scopus_paper_authors(doi):
    if not ELSEVIER_API_KEY:
        raise RuntimeError("Set ELSEVIER_API_KEY to query Scopus.")
    headers = {"X-ELS-APIKey": ELSEVIER_API_KEY, "Accept": "application/json"}
    try:
        data, from_cache = cached_get(SCOPUS_ABSTRACT + doi, params={"view": "META"},
                                      headers=headers, namespace="scopus_abstract")
    except requests.HTTPError as e:
        if e.response is not None and e.response.status_code == 404:   # DOI not indexed in Scopus
            return []
        raise
    if not from_cache:
        time.sleep(SCOPUS_SLEEP)
    resp = data.get("abstracts-retrieval-response", {})
    authors = as_list((resp.get("authors") or {}).get("author"))
    if not authors:
        authors = as_list(((resp.get("coredata") or {}).get("dc:creator") or {}).get("author"))
    out = []
    for a in authors:
        pref = a.get("preferred-name") or {}
        given = a.get("ce:given-name") or pref.get("ce:given-name") or a.get("ce:initials") or ""
        surname = a.get("ce:surname") or pref.get("ce:surname") or ""
        if a.get("@auid"):
            out.append({"auid": a["@auid"], "name": f"{given} {surname}".strip()})
    return out

### Decision rule (vote across papers)

Each checked paper votes for the Scopus ID of the author whose surname matches the laureate (if several co-authors share the surname, the given-name initials break the tie).

- `high`: the top ID got ≥ 3 votes and at least twice as many as any other ID.
- `medium`: the top ID got ≥ 2 votes and more than any other ID.
- `review`: a single vote, or a tie — check `scopus_id_all` (all IDs with their vote counts).
- `not found`: no checked paper had a matching author in Scopus.

Several IDs with votes often mean the laureate has **duplicate Scopus profiles**; they are all listed in `scopus_id_all`.

In [11]:
HIGH_CONFIDENCE_MATCHES = ("award", "exact", "first+last")

def vote(votes):
    if not votes:
        return None, "not found"
    ranked = votes.most_common()
    top_id, top = ranked[0]
    second = ranked[1][1] if len(ranked) > 1 else 0
    if top >= 3 and top >= 2 * second:
        return top_id, "high"
    if top >= 2 and top > second:
        return top_id, "medium"
    return top_id, "review"

def resolve(row):
    empty = {"openalex_id": None, "openalex_source": None, "papers_checked": 0, "evidence": None}
    # 1) Wikidata already has the Scopus ID
    if isinstance(row.get("wd_scopus_ids"), str):
        ids = row["wd_scopus_ids"].split(";")
        return {**empty, "scopus_id": ids[0], "scopus_id_all": row["wd_scopus_ids"], "source": "wikidata",
                "confidence": "high" if row["wd_match"] in HIGH_CONFIDENCE_MATCHES else "review"}
    # 2) OpenAlex author and papers
    author, how = find_openalex_author(row)
    if not author:
        return {**empty, "scopus_id": None, "scopus_id_all": None, "source": None, "confidence": "not found"}
    info = {"openalex_id": short_id(author["id"]), "openalex_source": how}
    if not ELSEVIER_API_KEY:
        return {**empty, **info, "scopus_id": None, "scopus_id_all": None, "source": None,
                "confidence": "not searched"}
    # 3) Scopus author IDs on those papers
    names = [n for n in (row.get("wd_label"), row["name"], author.get("display_name")) if isinstance(n, str)]
    papers = pick_papers(author["id"])
    votes, evidence = Counter(), []
    for p in papers:
        hits = matching_authors(names, scopus_paper_authors(p["doi"]))
        votes.update(hits)
        evidence.append(f"{p['doi']} -> {','.join(sorted(hits)) or '-'}")
    scopus_id, conf = vote(votes)
    if how == "name" and conf == "high":
        conf = "medium"   # the OpenAlex profile itself came from a name search
    return {**info, "scopus_id": scopus_id,
            "scopus_id_all": ";".join(f"{i}({n})" for i, n in votes.most_common()) or None,
            "source": "scopus:abstract" if scopus_id else None, "confidence": conf,
            "papers_checked": len(papers), "evidence": " | ".join(evidence) or None}

## Run

In [12]:
resolved = []
for _, row in tqdm(result.iterrows(), total=len(result)):
    try:
        resolved.append(resolve(row.to_dict()))
    except requests.HTTPError as e:
        resolved.append({"scopus_id": None, "source": None,
                         "confidence": f"error {e.response.status_code} ({e.response.url.split('?')[0]})"})

final = pd.concat([result.reset_index(drop=True), pd.DataFrame(resolved)], axis=1)
final["scopus_url"] = final["scopus_id"].map(
    lambda i: f"https://www.scopus.com/authid/detail.uri?authorId={i}" if isinstance(i, str) else None)

final.to_csv(OUTPUT_FILE, index=False)
print("Saved to", OUTPUT_FILE)
# when reading the file back, keep the IDs as text: pd.read_csv(OUTPUT_FILE, dtype=str)
final["confidence"].value_counts(dropna=False)

100%|██████████| 257/257 [00:55<00:00,  4.67it/s]

Saved to output/nobel_scopus_ids_free.csv


confidence
high            173
not searched     84
Name: count, dtype: int64

## Review

- `high`: Scopus ID curated in Wikidata, or the same ID found on ≥ 3 of the laureate's papers.
- `medium`: the ID found on ≥ 2 papers, or the OpenAlex profile came from a name search — worth spot-checking.
- `review`: uncertain; check `evidence` (which DOI gave which ID) and `scopus_url`.
- `not found` / `not searched` / `error …`: no profile, no API key, or an API failure. A 401/403 from `api.elsevier.com` means your key does not have access to the Abstract Retrieval API.

In [ ]:
cols = [c for c in ["category", "award_year"] if c in final] + [
    "name", "wd_label", "wd_match", "openalex_id", "openalex_source",
    "scopus_id", "scopus_id_all", "source", "confidence", "evidence", "scopus_url"]
final.loc[~final["confidence"].isin(["high"]), cols]